# 04 — Collective Behavior and Coordination

## Student Notebook

This notebook asks a harder question than the previous ones:

> do cells in the same local environment behave independently, or do nearby cells show coordinated signaling patterns?

This is a notebook about **collective behavior**. That does not necessarily mean direct cell-cell communication has been proven.
It means we are looking for patterns suggesting that cells in a shared neighborhood or site may behave in related ways.

We will explore:

- whether a cell resembles its local neighborhood at one time point,
- whether site-level averages rise and fall over time,
- whether observed neighbor similarity looks stronger than a shuffled baseline.

In [ ]:
from pathlib import Path
import os
import tempfile

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'mpl-config'))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from IPython.display import display

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)

ROOT = Path.cwd()
if not (ROOT / 'single-cell-tracks_exp1-6_noErbB2.csv.gz').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH = ROOT / '01-readme-experiment-description_2022-04-05.csv'

meta = pd.read_csv(META_PATH, encoding='utf-8-sig').rename(columns={'Site': 'Image_Metadata_Site'})
meta['Image_Metadata_Site'] = meta['Image_Metadata_Site'].astype(int)
site_to_mutation = meta.set_index('Image_Metadata_Site')['Mutation'].to_dict()
frame_to_min = float(meta['Acquisition_frequency_min'].iloc[0])

print('Collective analysis notebook ready.')

## What Counts as Collective Structure?

A collective pattern is one that lives at a level larger than one isolated cell.

For example, we may ask whether:

- neighboring cells have similar reporter values,
- site-level averages rise and fall together over time,
- spatial neighborhoods look more coordinated than expected by chance.

These are exploratory signatures, not final proof.
They help us decide whether it is worth building more careful statistical or mechanistic analyses later.

In [ ]:
collective_cols = [
    'Exp_ID', 'Image_Metadata_Site', 'Image_Metadata_T',
    'ERKKTR_ratio', 'FoxO3A_ratio',
    'objNuclei_Location_Center_X', 'objNuclei_Location_Center_Y'
]

sample = pd.read_csv(DATA_PATH, usecols=collective_cols, nrows=800_000)
sample['Mutation'] = sample['Image_Metadata_Site'].map(site_to_mutation)
sample['time_h'] = sample['Image_Metadata_T'] * frame_to_min / 60.0

site_counts = (
    sample.groupby(['Exp_ID', 'Image_Metadata_Site', 'Mutation'])
    .size()
    .reset_index(name='n_rows')
    .sort_values('n_rows', ascending=False)
)

chosen_site = site_counts.iloc[0]
exp_id = int(chosen_site['Exp_ID'])
site_id = int(chosen_site['Image_Metadata_Site'])
mutation = chosen_site['Mutation']
site_block = sample[(sample['Exp_ID'] == exp_id) & (sample['Image_Metadata_Site'] == site_id)].copy()

time_counts = site_block.groupby('Image_Metadata_T').size().sort_values(ascending=False)
selected_time = int(time_counts.index[0])
snapshot = site_block[site_block['Image_Metadata_T'] == selected_time].copy().reset_index(drop=True)

print(f'Using Exp_ID={exp_id}, Site={site_id}, Mutation={mutation}')
print(f'Selected time index: {selected_time}')
print('Cells in selected snapshot:', len(snapshot))

## Does a Cell Resemble Its Neighborhood?

One intuitive way to look for collective structure is to compare each cell with the average of its nearby neighbors.

If nearby cells tend to have similar ERK or FoxO values, that could reflect:

- shared local environment,
- local coordination,
- common input,
- or some mixture of these possibilities.

Again, this is exploratory. The goal is not to claim a mechanism too early, but to see whether neighborhood similarity is even visible in the data.

In [ ]:
coords = snapshot[['objNuclei_Location_Center_X', 'objNuclei_Location_Center_Y']].to_numpy()
tree = cKDTree(coords)
dists, _ = tree.query(coords, k=2)
radius = float(np.median(dists[:, 1]) * 3)
neighbor_ids = tree.query_ball_point(coords, r=radius)

neighbor_erk = []
neighbor_foxo = []
for i, ids in enumerate(neighbor_ids):
    ids = [j for j in ids if j != i]
    if ids:
        neighbor_erk.append(snapshot.loc[ids, 'ERKKTR_ratio'].mean())
        neighbor_foxo.append(snapshot.loc[ids, 'FoxO3A_ratio'].mean())
    else:
        neighbor_erk.append(np.nan)
        neighbor_foxo.append(np.nan)

snapshot['neighbor_ERK_mean'] = neighbor_erk
snapshot['neighbor_FoxO_mean'] = neighbor_foxo

erk_corr = snapshot[['ERKKTR_ratio', 'neighbor_ERK_mean']].dropna().corr().iloc[0, 1]
foxo_corr = snapshot[['FoxO3A_ratio', 'neighbor_FoxO_mean']].dropna().corr().iloc[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=snapshot, x='neighbor_ERK_mean', y='ERKKTR_ratio', scatter_kws={'alpha': 0.7, 's': 45}, line_kws={'color': 'black'}, ax=axes[0])
axes[0].set_title(f'ERK vs neighborhood mean (r = {erk_corr:.2f})')
axes[0].set_xlabel('Mean ERK among nearby cells')
axes[0].set_ylabel('Cell ERKKTR_ratio')

sns.regplot(data=snapshot, x='neighbor_FoxO_mean', y='FoxO3A_ratio', scatter_kws={'alpha': 0.7, 's': 45}, line_kws={'color': 'black'}, ax=axes[1])
axes[1].set_title(f'FoxO vs neighborhood mean (r = {foxo_corr:.2f})')
axes[1].set_xlabel('Mean FoxO among nearby cells')
axes[1].set_ylabel('Cell FoxO3A_ratio')

plt.tight_layout()

## Site-Level Coordination Over Time

Collective behavior can also appear as a property of the whole site rather than of one snapshot.

So next we look at site-level averages and spread over time.
This helps us ask whether the population inside a single field of view drifts or fluctuates together.

A useful distinction here is:

- the **site mean**, which captures the overall state of the local population,
- the **site standard deviation**, which captures heterogeneity among cells in that population.

In [ ]:
site_time = (
    site_block
    .groupby('Image_Metadata_T')
    .agg(
        n_cells=('ERKKTR_ratio', 'size'),
        ERK_mean=('ERKKTR_ratio', 'mean'),
        ERK_std=('ERKKTR_ratio', 'std'),
        FoxO_mean=('FoxO3A_ratio', 'mean'),
        FoxO_std=('FoxO3A_ratio', 'std'),
    )
    .reset_index()
)
site_time['time_h'] = site_time['Image_Metadata_T'] * frame_to_min / 60.0

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=site_time, x='time_h', y='ERK_mean', ax=axes[0], label='ERK mean')
sns.lineplot(data=site_time, x='time_h', y='FoxO_mean', ax=axes[0], label='FoxO mean')
axes[0].set_title('Site-level mean reporter signals over time')
axes[0].set_ylabel('Mean signal')

sns.lineplot(data=site_time, x='time_h', y='ERK_std', ax=axes[1], label='ERK std')
sns.lineplot(data=site_time, x='time_h', y='FoxO_std', ax=axes[1], label='FoxO std')
axes[1].set_title('Site-level heterogeneity over time')
axes[1].set_ylabel('Standard deviation across cells')
axes[1].set_xlabel('Time [h]')

plt.tight_layout()
display(site_time.head())

## Is Neighborhood Similarity More Than a Random Arrangement?

A useful reality check is to compare the observed neighborhood correlation with a shuffled baseline.

Here we keep the cell positions fixed, but randomly shuffle reporter values among cells.
That destroys any real relationship between value and neighborhood while preserving the same distribution of signal values.

If the observed correlation is much larger than shuffled values, that supports the idea that local similarity is not just an accident of the marginal distribution.

In [ ]:
observed = erk_corr
shuffled = []
valid_ids = [ids for ids in neighbor_ids]
values = snapshot['ERKKTR_ratio'].to_numpy()

for _ in range(200):
    permuted = np.random.permutation(values)
    neighbor_means = []
    for i, ids in enumerate(valid_ids):
        ids = [j for j in ids if j != i]
        if ids:
            neighbor_means.append(permuted[ids].mean())
        else:
            neighbor_means.append(np.nan)
    tmp = pd.DataFrame({'value': permuted, 'neighbor_mean': neighbor_means}).dropna()
    shuffled.append(tmp.corr().iloc[0, 1])

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(shuffled, bins=20, ax=ax, color='gray')
ax.axvline(observed, color='crimson', linestyle='--', linewidth=3, label=f'Observed r = {observed:.2f}')
ax.set_title('Observed neighborhood similarity vs shuffled baseline')
ax.set_xlabel('Correlation between ERK and neighborhood mean ERK')
ax.legend()
plt.tight_layout()

## What to Take Away

Collective behavior is a higher-level question than simple single-cell description.

This notebook introduces three useful habits:

- compare each cell with its neighborhood,
- track the state of a whole site through time,
- use a shuffled baseline to keep exploratory claims honest.

If these patterns remain interesting after this first pass, the next step would be more formal analysis:
for example, repeated tests across many sites, more careful null models, and definitions of neighborhoods that depend on the biological question.